In [ ]:
!pip install --upgrade "mlflow>=3.1"
!pip install pandas
!pip install scikit-learn
!pip install boto3
!pip install python-dotenv

  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached gunicorn-25.1.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached databricks_sdk-0.96.0-py3-none-any.whl.metadata (40 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached mako-1.3.10-py3-none-any.whl.metadata (2.9 kB)
  Using cached google_auth-2.48.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached werkzeug-3.1.6-py3-none-any.whl.metadata (4.0 kB)
  Using cached g

In [ ]:
import pandas as pd
import s3fs
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
from datetime import datetime
import json
import shutil
from dotenv import load_dotenv

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [ ]:
# =======================================================
# TODO MIEMBRO DEL EQUIPO: MODIFICAR SOLO ESTA VARIABLE
# =======================================================
EXPERIMENT_FOLDER = "Exp15_Todo_Activo"

# ---------- S3 ----------
S3_BUCKET = os.getenv("S3_BUCKET_NAME")
TRAIN_PATH = f"{S3_BUCKET}/data/processed/Spacy/{EXPERIMENT_FOLDER}/train_parquet"

# Usamos 'models_experiments' para NO disparar la Lambda a Producción
DEST_BUCKET = f"{S3_BUCKET}/data/models_experiments/FaseA_Ablacion"
DATASET_VERSION = EXPERIMENT_FOLDER

# ---------- MLflow ----------
S3_BUCKET_MLFLOW = os.getenv("S3_BUCKET_MLFLOW")
EC2_HOST = os.getenv("EC2_HOST")
MLFLOW_TRACKING_URI = f"{EC2_HOST}"
MLFLOW_ARTIFACT_BUCKET = f"{S3_BUCKET_MLFLOW}"
EXPERIMENT_NAME = "Sentimientos403_FaseA" 
REGISTERED_MODEL_NAME = "Sentimientos403_Ablacion"
MODEL_TYPE = "Ablacion_NB"
TEAM = "NPL"

# ---------- Modelo ----------
TEXT_COLUMN = "ablation_text"
TARGET_COLUMN = "label"
MAX_FEATURES = 5000
AUTOR_EXPERIMENTO = "Daniel Varela"

In [105]:
fs = s3fs.S3FileSystem()

def load_parquet_from_s3(prefix: str) -> pd.DataFrame:
    files = fs.ls(prefix)
    df_list = [pd.read_parquet(f"s3://{file}", filesystem=fs) for file in files]
    return pd.concat(df_list, ignore_index=True)

print("Cargando datos de entrenamiento...")
full_train_df = load_parquet_from_s3(TRAIN_PATH)

# Agrega esta línea para ver los nombres exactos:
print("Columnas disponibles:", full_train_df.columns.tolist())

Cargando datos de entrenamiento...
Columnas disponibles: ['text', 'label', 'clean_text', 'ablation_text']


In [106]:
X = full_train_df[TEXT_COLUMN]
y = full_train_df[TARGET_COLUMN]

print("Dividiendo en Entrenamiento (80%) y Validación (20%)...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Dividiendo en Entrenamiento (80%) y Validación (20%)...


In [107]:
pipeline = Pipeline([
    ("vectorizer", CountVectorizer(max_features=MAX_FEATURES)),
    ("model", MultinomialNB())
])

In [108]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    artifact_location = f"{MLFLOW_ARTIFACT_BUCKET}/{EXPERIMENT_NAME}/"
    print(f"Creando experimento '{EXPERIMENT_NAME}' con artefactos en {artifact_location}")
    mlflow.create_experiment(name=EXPERIMENT_NAME, artifact_location=artifact_location)
else:
    print(f"Utilizando experimento existente '{EXPERIMENT_NAME}'")

mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()

Utilizando experimento existente 'Sentimientos403_FaseA'


In [109]:
# Usamos el nombre de la carpeta como nombre del Run
with mlflow.start_run(run_name=EXPERIMENT_FOLDER):

    mlflow.set_tag("Autor", AUTOR_EXPERIMENTO)
    mlflow.set_tag("dataset_version", DATASET_VERSION)
    mlflow.set_tag("model_type", MODEL_TYPE)
    mlflow.set_tag("team", TEAM)

    print(f"Entrenando modelo para {EXPERIMENT_FOLDER}...")
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Macro: {f1:.4f}")

    mlflow.log_params({
        "model_type": "MultinomialNB",
        "vectorizer": "CountVectorizer",
        "max_features": MAX_FEATURES,
        "spacy_experiment": EXPERIMENT_FOLDER
    })

    mlflow.log_metrics({
        "accuracy": acc,
        "f1_macro": f1
    })

    signature = infer_signature(X_train, pipeline.predict(X_train))

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="AblationModel",
        signature=signature,
        input_example=X_train.iloc[:3].to_frame(),
        registered_model_name=REGISTERED_MODEL_NAME
    )

print("Modelo registrado correctamente.")

Entrenando modelo para Exp15_Todo_Activo...
Accuracy: 0.7533
F1 Macro: 0.7532


2026/03/06 02:33:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/06 02:33:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Sentimientos403_Ablacion' already exists. Creating a new version of this model...
2026/03/06 02:33:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Sentimientos403_Ablacion, version 15
Created version '15' of model 'Sentimientos403_Ablacion'.


🏃 View run Exp15_Todo_Activo at: http://ec2-52-21-111-46.compute-1.amazonaws.com:5000/#/experiments/9/runs/350bdbe488434001aa1aacdf90f886e8
🧪 View experiment at: http://ec2-52-21-111-46.compute-1.amazonaws.com:5000/#/experiments/9
Modelo registrado correctamente.


In [110]:
local_tmp = "tmp_champion_model"

########################### MODEL CARD ######################
model_card = {
    "model_name": EXPERIMENT_FOLDER,
    "description": "Modelo de evaluación de ablación (Phase A) usando BoW y Naive Bayes.",
    "author": AUTOR_EXPERIMENTO,
    "dataset_version": DATASET_VERSION,
    "training_data_path": TRAIN_PATH,
    "version": str(datetime.now()),
    "performance": {
        "validation_accuracy": round(acc, 4),
        "validation_f1_macro": round(f1, 4)
    },
    "intended_use": "Encontrar la mejor limpieza de texto (Ablación) para el proyecto.",
    "limitations": [
        "Dependiente de la configuración específica de spaCy"
    ]
}

################################# MLFLOW CHECKIN #############################
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)

print(f"Nueva versión registrada: {latest_version}")

try:
    champion_info = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
    champion_version = int(champion_info.version)
    champion_run = client.get_run(champion_info.run_id)
    champion_f1 = champion_run.data.metrics.get("f1_macro", 0)
    print(f"Champion actual: v{champion_version} | F1={champion_f1:.4f}")
except Exception:
    champion_f1 = -1
    print("No existe champion aún.")

print(f"Nuevo modelo F1: {f1:.4f}")

if f1 > champion_f1:
    ################## Promover en MLflow #################
    client.set_registered_model_alias(name=REGISTERED_MODEL_NAME, alias="champion", version=latest_version)
    client.set_model_version_tag(name=REGISTERED_MODEL_NAME, version=latest_version, key="status", value="champion")

    ############## Guardar Model Card en S3 ############
    with fs.open(f"{DEST_BUCKET}/model_card.json", "w") as f:
        f.write(json.dumps(model_card, indent=4))
    
    ########################### Sincronizar el .pkl del Champion en S3 #######################
    mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{REGISTERED_MODEL_NAME}@champion", dst_path=local_tmp)
    fs.put(local_tmp, DEST_BUCKET, recursive=True)
    shutil.rmtree(local_tmp)
    print("¡Nuevo Champion! Modelo y Model Card sincronizados en el bucket de experimentos.")

else:
    print("El champion actual sigue siendo mejor. No se sobrescriben archivos en S3.")

Nueva versión registrada: 15
Champion actual: v4 | F1=0.7748
Nuevo modelo F1: 0.7532
El champion actual sigue siendo mejor. No se sobrescriben archivos en S3.
